# Team SentiMix — ABSA for Hindi-English Code-Mixed E-commerce Reviews
**Starter Research & Experimentation Notebook**

This notebook implements the foundational ABSA pipeline matching Objectives 1–5:
1. Data collection & multi-source ingestion
2. Preprocessing (cleaning, normalization, tokenization, emoji preservation)
3. Aspect extraction (rule-based domain taxonomy)
4. Sentiment classification (Lexicon baseline + Model fine-tuning)
5. Evaluation & Error Analysis


In [1]:
# ---- 1. Environment setup ----
import os
import sys
import re
import ast
import emoji
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath("."))
try:
    import src.torch_fix
except Exception:
    pass

print("PyTorch Version:", torch.__version__)
print("GPU available  :", torch.cuda.is_available())


PyTorch Version: 2.10.0+cpu
GPU available  : False


## 2. Load Dataset
Loads local e-commerce datasets from `data/` or downloads directly if in Colab.


In [2]:
def get_dataset_path(filename="Multilingual_Meesho_Review_Dataset.csv"):
    if os.path.exists(filename):
        return filename
    p = os.path.join("data", filename)
    if os.path.exists(p):
        return p
    return filename

meesho_file = get_dataset_path("Multilingual_Meesho_Review_Dataset.csv")
df = pd.read_csv(meesho_file)
df = df.dropna(subset=["comments"]).rename(columns={"comments": "review_text"}).reset_index(drop=True)
print(f"Loaded {len(df)} e-commerce reviews from {meesho_file}")
display(df.head(5))


Loaded 1005 e-commerce reviews from data\Multilingual_Meesho_Review_Dataset.csv
   id                                       product_name        reviewer       date  ratings                                                                                                                                                                                                                                                                                                                             review_text mapped_sentiment images  source  helpful category
0   1  Womens Ethnic Embroidery Top Pant And Dupatta Set     Meesho User  12-Sep-25        5                                                                                                                                                                                                                                                           very nice quality of this suit it is having three part e suit 2 jama 3 Chunri        very good    NaN  Mee

## 3. Filter for Genuinely Code-Mixed (Hinglish) Reviews
We isolate code-mixed reviews that mix Romanized Hindi, Devanagari Hindi, and English vocabulary.


In [3]:
HINDI_ROMAN_WORDS = set([
    "hai","nahi","nahin","bahut","bohot","accha","acha","achha","bekar","sahi","paisa","paise",
    "product","mast","badhiya","theek","kharab","bura","kaam","pasand","zyada","kam",
    "lagta","lagti","hoga","mila","milta","dikkat","problem","bilkul","lekin","aur"
])

def is_code_mixed(text):
    if not isinstance(text, str):
        return False
    # Check Devanagari Unicode
    if re.search(r"[ऀ-ॿ]", text):
        return True
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    has_hindi = any(t in HINDI_ROMAN_WORDS for t in tokens)
    has_english = any(t not in HINDI_ROMAN_WORDS and len(t) > 2 for t in tokens)
    return has_hindi and has_english

df["is_code_mixed"] = df["review_text"].apply(is_code_mixed)
code_mixed_df = df[df["is_code_mixed"]].copy().reset_index(drop=True)
print(f"Kept {len(code_mixed_df)} / {len(df)} reviews as code-mixed ({len(code_mixed_df)/len(df)*100:.1f}%)")
display(code_mixed_df.head(5))


Kept 501 / 1005 reviews as code-mixed (49.9%)
   id                                       product_name         reviewer       date  ratings                                                                                                                              review_text mapped_sentiment images  source  helpful category  is_code_mixed
0   5  Womens Ethnic Embroidery Top Pant And Dupatta Set           Garima   6-Aug-25        5           Fabric bahut achcha hai aur design bhi bahut achcha hai but iske niche mein Aisa Hi design aur de diya hota to aur achcha hota        very good    NaN  Meesho       58  Fashion           True
1   7  Womens Ethnic Embroidery Top Pant And Dupatta Set      Arhaan Alam  25-Jun-25        5          Bhut hi pyara suit aaya h ,🥰😘😚😍😘🥰😍ise aap log buy kr skte h qki mughe bhut hi pyara or comfortable suit Mila h thank you meshoo        very good    NaN  Meesho       21  Fashion           True
2   9  Womens Ethnic Embroidery Top Pant And Dupatta Set  Kalpana 

## 4. Preprocessing Pipeline (Objective 2)
Cleaning + text normalization with emoji demojization.


In [4]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"(.){2,}", r"", text)
    text = emoji.demojize(text, delimiters=(" :", ": "))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return re.findall(r"[\w']+|[.,!?;]", text)

code_mixed_df["clean_text"] = code_mixed_df["review_text"].apply(clean_text)
code_mixed_df["tokens"] = code_mixed_df["clean_text"].apply(tokenize)
display(code_mixed_df[["review_text", "clean_text", "tokens"]].head(5))


                                                                                                                               review_text                                                                                                                                                                                                                                                                                                                 clean_text                                                                                                                                                                                                                                                                                                                                            tokens
0           Fabric bahut achcha hai aur design bhi bahut achcha hai but iske niche mein Aisa Hi design aur de diya hota to aur achcha hota                                                                      

## 5. Aspect Extraction — Rule-Based Taxonomy (Objective 3)
Extracts aspect terms matching 6 major e-commerce product dimensions.


In [5]:
ASPECT_VOCAB = {
    "battery": ["battery", "backup", "charge", "charging"],
    "camera": ["camera", "photo", "picture", "selfie", "lens"],
    "price": ["price", "paisa", "paise", "cost", "expensive", "sasta", "mehenga", "worth"],
    "delivery": ["delivery", "shipping", "deliver", "late", "time pe", "packaging"],
    "quality": ["quality", "build", "material", "finish", "fabric", "kapda", "silai"],
    "display": ["display", "screen", "resolution", "amoled"],
}

def extract_aspects(tokens):
    text = " ".join(tokens).lower()
    found = []
    for aspect, keywords in ASPECT_VOCAB.items():
        if any(kw in text for kw in keywords):
            found.append(aspect)
    return found if found else ["product"]

code_mixed_df["aspects"] = code_mixed_df["tokens"].apply(extract_aspects)
display(code_mixed_df[["clean_text", "aspects"]].head(10))


                                                                                                                                                                                                                                                                                                                  clean_text           aspects
0                                                                                                                                                                                             Fabric bahut achcha hai aur design bhi bahut achcha hai but iske niche mein Aisa Hi design aur de diya hota to aur achcha hota         [quality]
1  Bhut hi pyara suit aaya h , :smiling_face_with_hearts: :face_blowing_a_kiss: :kissing_face_with_closed_eyes: :smiling_face_with_heart-eyes: :face_blowing_a_kiss: :smiling_face_with_hearts: :smiling_face_with_heart-eyes: ise aap log buy kr skte h qki mughe bhut hi pyara or comfortable suit Mila h thank you meshoo         [prod

## 6. Sentiment Baseline — Lexicon-Based Model (Objective 4, Model 1)
Rule-based lexicon polarity lookup for Hindi, Hinglish, and English tokens.


In [6]:
POSITIVE_WORDS = set(["accha","acha","achha","badhiya","mast","sahi","good","great","love","best","nice","awesome","super","vibrant","perfect"])
NEGATIVE_WORDS = set(["bekar","kharab","bura","bad","worst","poor","problem","dikkat","waste","late","ghatiya","barbad","faded","damaged"])

def lexicon_sentiment(text):
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    pos = sum(t in POSITIVE_WORDS for t in tokens)
    neg = sum(t in NEGATIVE_WORDS for t in tokens)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    return "neutral"

code_mixed_df["lexicon_sentiment"] = code_mixed_df["clean_text"].apply(lexicon_sentiment)
print("Sentiment distribution predicted by Lexicon Baseline:")
print(code_mixed_df["lexicon_sentiment"].value_counts())


Sentiment distribution predicted by Lexicon Baseline:
lexicon_sentiment
positive    256
neutral     229
negative     16
Name: count, dtype: int64


## 7. Train & Evaluate ABSA Classifier (Objective 4 & 5)
Train and evaluate TF-IDF and cross-lingual classifiers on labeled ABSA pairs.


In [7]:
from src.models.tfidf_classifier import TFIDFLogisticABSAModel
from src.dataset_loader import ABSADatasetLoader
from src.evaluation import ModelEvaluator

loader = ABSADatasetLoader(data_dir="data")
datasets = loader.load_all_datasets()
train_df = datasets["combined_train"]
test_df = datasets["combined_test"]

print(f"Training on {len(train_df)} samples, testing on {len(test_df)} samples...")

tfidf_model = TFIDFLogisticABSAModel()
tfidf_model.train(train_df, val_df=test_df)

eval_res = ModelEvaluator.evaluate_model_on_test_set(tfidf_model, test_df)

print("=======================================================")
print("TF-IDF + LOGISTIC REGRESSION ABSA EVALUATION RESULTS")
print("=======================================================")
print(f"Accuracy : {eval_res['accuracy']*100:.2f}%")
print(f"Macro F1 : {eval_res['f1_macro']*100:.2f}%")
print(f"Latency  : {eval_res['latency_ms_per_sample']:.2f} ms / review")


=== Dataset Summary Loaded ===
laptop_train        :  2354 aspect-sentiment pairs
laptop_test         :   654 aspect-sentiment pairs
restaurant_train    :  3692 aspect-sentiment pairs
restaurant_test     :  1134 aspect-sentiment pairs
meesho_train        :  1184 aspect-sentiment pairs
meesho_test         :   297 aspect-sentiment pairs
flipkart_train      :  1217 aspect-sentiment pairs
flipkart_test       :   305 aspect-sentiment pairs
amazon_train        :   693 aspect-sentiment pairs
amazon_test         :   174 aspect-sentiment pairs
augmented_train     :   720 aspect-sentiment pairs
augmented_test      :   180 aspect-sentiment pairs
combined_train      :  9860 aspect-sentiment pairs
combined_test       :  2744 aspect-sentiment pairs
Training on 9860 samples, testing on 2744 samples...
Training TF-IDF + Logistic Regression on 9860 samples...
Train Accuracy: 0.9009 | Val Accuracy: 0.7387
TF-IDF + LOGISTIC REGRESSION ABSA EVALUATION RESULTS
Accuracy : 73.87%
Macro F1 : 65.18%
Latency  :

## 8. Summary & Next Steps
- Preprocessing and aspect extraction operational across Hindi-English code-mixed reviews.
- Baseline models trained and evaluated.
- To explore the full 6-model benchmark and live dashboard, run `NLP_ABSA.ipynb` and `server.py`.
